# HITL-IoT v1.1 — Leakage-safe Kaggle quickstart
This is a **tutorial model**, **not** a reproduction of the nine-model results in the accepted manuscript. The 10,227 oversight profiles are **generated**, not human analyst annotations.

Use the dataset in Kaggle by clicking **New Notebook** on the dataset page; Kaggle will attach the current version under `/kaggle/input/`.


In [ ]:
from pathlib import Path
import json, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score, f1_score, matthews_corrcoef, confusion_matrix

local_base = Path("data/v1.1")
if (local_base / "HITL-IoT_dataset_v1.1.csv").exists():
    base = local_base
else:
    matches = list(Path("/kaggle/input").rglob("HITL-IoT_dataset_v1.1.csv"))
    assert len(matches) == 1, f"Expected one v1.1 dataset; found {matches}"
    base = matches[0].parent
df = pd.read_csv(base / "HITL-IoT_dataset_v1.1.csv")
splits = pd.read_csv(base / "HITL-IoT_splits_v1.1.csv")
policy = json.loads((base / "HITL-IoT_feature_policy_v1.1.json").read_text())
assert len(df) == 127_845 and df.shape[1] == 60
print("Source:", base, "Data shape:",df.shape)


## Use the exact frozen splits and 39 permitted input fields
Do not include generator scores, synthetic oversight metadata, target labels or identifiers in the primary IDS model.


In [ ]:
assert splits["flow_id"].is_unique and df["flow_id"].is_unique
df = df.merge(splits[["flow_id", "temporal_split"]], on="flow_id", validate="one_to_one")
features = policy["track1_primary_flow_only"]
assert len(features) == 39 and all(c in df.columns for c in features)
sets = {s:df.loc[df.temporal_split.eq(s)] for s in ("train","validation","test")}
print({s:len(x) for s,x in sets.items()})
assert [len(sets[s]) for s in ("train","validation","test")] == [89492, 19176, 19177]
X_train, y_train = sets["train"][features], sets["train"]["is_attack"].astype(int)
X_val, y_val = sets["validation"][features], sets["validation"]["is_attack"].astype(int)


In [ ]:
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in features if c not in num_cols]
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([( "impute", SimpleImputer(strategy="most_frequent")),
                       ("encode", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])
tutorial_model = Pipeline([("prep",prep), ("clf",DecisionTreeClassifier(max_depth=10, min_samples_leaf=15, random_state=42))])
tutorial_model.fit(X_train, y_train)
pred = tutorial_model.predict(X_val)
print("Validation balanced accuracy:", round(balanced_accuracy_score(y_val,pred),4))
print("Validation Macro-F1:", round(f1_score(y_val,pred,average="macro"),4))
print("Validation MCC:", round(matthews_corrcoef(y_val,pred),4))
print("Validation confusion matrix:\n",confusion_matrix(y_val,pred))


## Scientific interpretation
- The example is for usability, not leaderboard comparison. It uses a different model, preprocessing and configuration than Table 7.
- The held-out test set is intentionally untouched; finalize the model using **validation only** before one final test.
- Do **not** claim human oversight outcomes from generated high/medium/low simulation profiles.
- For scholarly reuse, cite the published article and the verified version-specific Zenodo release identifier.
